# Scrape Multiple Listing Pages

Walk through several numbered listing pages (page 1, 2, 3...) and collect every author link, removing duplicates. The demo uses the public practice site quotes.toscrape.com and waits between requests.

## How to run
Set `NUM_PAGES` and run all cells. Links are saved to `links.csv`.

_Rewritten from my original Colab notebook: `dwonload with 2 like pages.ipynb`._

In [ ]:
!pip install -q requests beautifulsoup4 pandas

In [ ]:
BASE_URL = "https://quotes.toscrape.com/"
NUM_PAGES = 3

In [ ]:
import time
import urllib.robotparser
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "learning-scraper/1.0 (educational project)"}
DELAY_SECONDS = 1.5

robots = urllib.robotparser.RobotFileParser()
robots.set_url(urljoin(BASE_URL, "/robots.txt"))
robots.read()

def get_soup(url):
    """Download a page politely and return it parsed with BeautifulSoup."""
    if not robots.can_fetch(HEADERS["User-Agent"], url):
        print("robots.txt does not allow:", url)
        return None
    time.sleep(DELAY_SECONDS)  # be gentle with the server
    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

## Loop over pages
Page 1 is the base URL; later pages follow the pattern `/page/N/`.

In [ ]:
author_links = []

for page in range(1, NUM_PAGES + 1):
    url = BASE_URL if page == 1 else f"{BASE_URL}page/{page}/"
    print("Scanning:", url)
    soup = get_soup(url)
    if soup is None:
        continue
    for a in soup.select("div.quote a[href^='/author/']"):
        link = urljoin(BASE_URL, a["href"])
        if link not in author_links:   # skip duplicates
            author_links.append(link)

print("Total unique links:", len(author_links))

In [ ]:
import pandas as pd

df = pd.DataFrame({"url": author_links})
df.to_csv("links.csv", index=False)
df.head()